In [1]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score

# 1. Load the dataset directly from a public URL
url = "https://raw.githubusercontent.com/justmarkham/pycon-2016-tutorial/master/data/sms.tsv"
df = pd.read_csv(url, sep='\t', header=None, names=['label', 'message'])

# 2. Look at the data (we have 'ham' and 'spam')
print("Dataset shape:", df.shape)
print(df['label'].value_counts())
display(df.head())

Dataset shape: (5572, 2)
label
ham     4825
spam     747
Name: count, dtype: int64


,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [2]:
# Split data (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(df['message'], df['label'], test_size=0.2, random_state=42)

# TF-IDF Vectorizer (text -> numbers)
vec = TfidfVectorizer()
X_train_tfidf = vec.fit_transform(X_train)
X_test_tfidf = vec.transform(X_test)

# Train the model
model = MultinomialNB()
model.fit(X_train_tfidf, y_train)

# Measure accuracy
pred = model.predict(X_test_tfidf)
print(f"Test Accuracy: {accuracy_score(y_test, pred):.4f}")
print(f"Precision (spam): {precision_score(y_test, pred, pos_label='spam'):.4f}")
print(f"Recall (spam): {recall_score(y_test, pred, pos_label='spam'):.4f}")

Test Accuracy: 0.9668
Precision (spam): 1.0000
Recall (spam): 0.7517


In [3]:
# 5 tricky messages designed to fool the model
tricky_tests = [
    "Cl@im your fr33 g1ft card now",                # 1. Spelling trick (spam)
    "Not free, just confirming lunch at 1pm",       # 2. Negation trick (ham)
    "Congratulations! You won a $1000 Walmart gift", # 3. Common spam words (spam)
    "Hey, are we still meeting at the library?",    # 4. Normal text (ham)
    "URGENT: Your account is locked. Click here to verify.", # 5. Business tone (spam)
]

# What the model actually predicts
predictions = model.predict(vec.transform(tricky_tests))

# What the correct answer should be
actual_labels = ["spam", "ham", "spam", "ham", "spam"]

# Why we think it failed
reasons = [
    "Spelling tricks ('fr33', 'g1ft') mean the words are missing from the vocabulary.",
    "The model saw 'free' and ignored the word 'Not'.",
    "This might actually be caught because it uses strong training words like 'gift' and 'won'.",
    "Should be correctly classified as 'ham'.",
    "The model might miss this if 'verify' or 'locked' weren't frequent in the training data."
]

# Create a results table
results_df = pd.DataFrame({
    "My Message": tricky_tests,
    "Model Said": predictions,
    "Should Be": actual_labels,
    "Why it failed/worked": reasons
})

display(results_df)

,My Message,Model Said,Should Be,Why it failed/worked
0,Cl@im your fr33 g1ft card now,ham,spam,"Spelling tricks ('fr33', 'g1ft') mean the word..."
1,"Not free, just confirming lunch at 1pm",ham,ham,The model saw 'free' and ignored the word 'Not'.
2,Congratulations! You won a $1000 Walmart gift,spam,spam,This might actually be caught because it uses ...
3,"Hey, are we still meeting at the library?",ham,ham,Should be correctly classified as 'ham'.
4,URGENT: Your account is locked. Click here to ...,ham,spam,The model might miss this if 'verify' or 'lock...
